# Old Faithful

![Old Faithful](https://thumb.wikimedia.org/wikipedia/commons/thumb/5/51/Yellowstone_National_Park_%28WY%2C_USA%29%2C_Old_Faithful_Geyser_--_2022_--_2599.jpg/960px-Yellowstone_National_Park_%28WY%2C_USA%29%2C_Old_Faithful_Geyser_--_2022_--_2599.jpg)

Old Faithful is a geyser in Yellowstone National Park, Wyoming, USA, known for erupting at fairly regular intervals. Visitors can wait around for the next eruption, which suggests a practical question: how long until it erupts again, and does the length of the last eruption help predict that?

## The data

The data record 272 eruptions. Each row has two measurements, both in minutes:

| Column | Meaning |
|---|---|
| `eruptions` | Duration of the eruption |
| `waiting` | Waiting time until the **next** eruption |

This is the `faithful` dataset from R's `datasets` package. R's documentation cites Härdle (1991), *Smoothing Techniques with Implementation in S*, and Azzalini and Bowman (1990), "A look at some data on the Old Faithful geyser", *Applied Statistics* 39, 357–365.

A few things worth knowing before you analyse it:

- Durations such as `3.333` and `2.283` look like minutes and seconds converted to decimal minutes (3 min 20 s, 2 min 17 s), so the values are not arbitrary real numbers.
- Eruptions happen in sequence in time, so consecutive observations may not be independent. The exercises mostly treat the rows as independent; keep this in mind as a limitation.
- Take a good look at the distribution of each variable before modelling anything. What you find affects every question below.

In this package, `load_faithful_data()` returns a list of `(eruptions, waiting)` tuples. The cells below wrap them in a data frame so you can start exploring.

Work through this notebook alongside the [exercise instructions](README.md).

In [ ]:
from itertools import islice

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats

from exercises._shared.loaders import load_faithful_data
from exercises.fitting_a_mixture.mixture_models import (
    MultivariateGaussianMixtureParameters,
    animate_faithful_mixtures,
    em_steps,
    fit_faithful_gaussian_mixture,
    multivariate_soft_expectation_step,
    multivariate_soft_maximization_step,
    plot_faithful_mixture,
)

rng = np.random.default_rng(2025)

faithful = pd.DataFrame(load_faithful_data(), columns=["eruptions", "waiting"])
faithful.head(10)

## Waiting time prediction

This short diagnostic starts the Old Faithful EM application. Fit
a straight line (for example with `numpy.polyfit` or `scipy.stats.linregress`),
interpret the intercept and slope in minutes, calculate $R^2$, and predict the
wait after a 2-minute and a 4-minute eruption. Plot the data with the fitted
line, and plot the residuals against the fitted values. Is a single line a good
model? What do the plots suggest about the data? This is a model diagnostic, not a held-out prediction evaluation; use it to motivate the mixture in the next section.

In [ ]:
# TODO: fit waiting time against eruption duration with a straight line.
# Report R^2 and predictions, then plot the fitted line and the residuals vs fitted values.

## Gaussian mixture

Revisit the questions your regression plots raised. Does the bivariate data
look like one cloud of points, or more? Fit the two-component bivariate Gaussian
mixture with `fit_faithful_gaussian_mixture()`, starting from the rough values in
the cell below (these are the ones the tests use). Plot the starting and fitted
mixtures with `plot_faithful_mixture`, which colours each eruption by its most
likely component, and compare the fitted log-likelihood
with the reference value in the tests (about -1130.27). Does the structure the
mixture finds explain what you saw in the regression residuals? EM can converge
to local optima, so explain what the starting values and the fitted result show.

If you're so inclined, animate the convergence. Your `em_steps` yields the
parameters after each iteration, starting with `initial`, so
`list(islice(em_steps(...), n))` gives a history to show with
`HTML(animate_faithful_mixtures(history).to_jshtml())`, using `HTML` from
`IPython.display`. It should resemble the principal example on the Wikipedia
page for the EM algorithm.

As an extension, revisit the prediction: assign each eruption to a cluster using
its duration alone (so the waiting time being predicted does not leak into the
predictor), fit a separate line within each cluster, and compare prediction
errors with the single line (for example by cross-validation).

In [ ]:
initial = MultivariateGaussianMixtureParameters(
    mean1=np.array([2.8, 75.0]),
    covariance1=np.array([[0.8, 7.0], [7.0, 70.0]]),
    mean2=np.array([3.6, 58.0]),
    covariance2=np.array([[0.8, 7.0], [7.0, 70.0]]),
    weight1=0.5,
)
# TODO: fit the bivariate Gaussian mixture with fit_faithful_gaussian_mixture(initial).
# Plot the starting and fitted mixtures with plot_faithful_mixture.